# 10 - Image : Optimisation et Modèles Alternatifs

## Objectif de ce notebook
**Améliorer** la baseline image du notebook 09 en testant : rééquilibrage des classes (class weights), optimisation des hyperparamètres et modèles de type boosting (XGBoost, LightGBM, CatBoost).

**Prérequis** : Exécuter le notebook 09 pour disposer des features image en cache, du label encoder et du modèle baseline dans `models/`.

**Livrable** : Meilleur modèle après optimisation, sauvegardé pour le notebook 11.

---

## Plan
1. Chargement des features et du modèle baseline (notebook 09)
2. Gestion du déséquilibre (class weights)
3. Modèles avancés (XGBoost, LightGBM, CatBoost)
4. Comparaison complète et sauvegarde du meilleur modèle

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = str(os.cpu_count() or 4)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import pickle
import warnings
warnings.filterwarnings('ignore')

sys.path.append(str(Path('../').resolve()))

from src.modeling import BaselineModels, AdvancedModels
from src.optimization import create_class_weights
from src.evaluation import evaluate_model, print_classification_report, plot_confusion_matrix

from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Bibliothèques importées")

✅ Bibliothèques importées


In [2]:
ROOT = Path('../').resolve()
MODELS_DIR = ROOT / 'models'
cache = np.load(MODELS_DIR / 'image_features_cache.npz', allow_pickle=True)
X_features = cache['X']
y_labels = cache['y']

with open(MODELS_DIR / 'label_encoder_image.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

y_encoded = label_encoder.transform(y_labels)

indices = np.arange(len(y_labels))
train_idx, val_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y_encoded
)
X_train_split = X_features[train_idx]
X_val_split = X_features[val_idx]
y_train = y_encoded[train_idx]
y_val = y_encoded[val_idx]

# Charger la baseline du NB09
baseline_files = list(MODELS_DIR.glob('image_*_27_classes_baseline.pkl'))
if baseline_files:
    baseline_model = BaselineModels.load_model(baseline_files[0])
    y_pred_baseline = baseline_model.predict(X_val_split)
    metrics_baseline = evaluate_model(y_val, y_pred_baseline)
else:
    baseline_model = LinearSVC(max_iter=2000, random_state=42, dual=False)
    baseline_model.fit(X_train_split, y_train)
    y_pred_baseline = baseline_model.predict(X_val_split)
    metrics_baseline = evaluate_model(y_val, y_pred_baseline)

print(f"✅ Données : train {X_train_split.shape[0]:,} | val {X_val_split.shape[0]:,}")
print(f"   Classes : {len(label_encoder.classes_)}")
print(f"   Baseline F1-macro : {metrics_baseline['f1_macro']:.4f}")

✅ Données : train 35,975 | val 8,994
   Classes : 27
   Baseline F1-macro : 0.5171


## 2. Modèles avancés (XGBoost, LightGBM, CatBoost)

Les class weights sont désormais intégrés dans le notebook 09 (baselines). Ici, on les calcule pour les passer aux modèles avancés.

In [3]:
class_weights = create_class_weights(y_train, method='balanced')
n_classes = len(class_weights)
w_vals = list(class_weights.values())
print(f"27 classes : {n_classes} classes, poids min={min(w_vals):.3f} max={max(w_vals):.3f} ratio={max(w_vals)/min(w_vals):.1f}x")

✅ Poids de classes calculés (balanced)
   Nombre de classes : 27
   Poids min : 0.4347
   Poids max : 18.5057
   Ratio max/min : 42.57
27 classes : 27 classes, poids min=0.435 max=18.506 ratio=42.6x


### Entraînement des modèles avancés

In [4]:
advanced_models = AdvancedModels(random_state=42)
advanced_models.create_advanced_models(class_weights=class_weights)
trained_advanced = advanced_models.train_all_models(X_train_split, y_train)

print("✅ Modèles avancés entraînés (27 classes)")

🔄 Entraînement de Random Forest (optimized)...
✅ Random Forest (optimized) entraîné avec succès
🔄 Entraînement de Gradient Boosting...
✅ Gradient Boosting entraîné avec succès
🔄 Entraînement de Logistic Regression (weighted)...
✅ Logistic Regression (weighted) entraîné avec succès
🔄 Entraînement de XGBoost...
✅ XGBoost entraîné avec succès
🔄 Entraînement de LightGBM...
✅ LightGBM entraîné avec succès
🔄 Entraînement de CatBoost...
✅ CatBoost entraîné avec succès
✅ Modèles avancés entraînés (27 classes)


## 3. Comparaison et sauvegarde

In [5]:
all_results = []

all_results.append({
    'Model': 'Baseline (NB09)',
    'F1_macro': metrics_baseline['f1_macro'],
    'F1_weighted': metrics_baseline['f1_weighted']
})

for name in trained_advanced:
    yp = advanced_models.predict(name, X_val_split)
    m = evaluate_model(y_val, yp)
    all_results.append({
        'Model': name,
        'F1_macro': m['f1_macro'],
        'F1_weighted': m['f1_weighted']
    })

results_df = pd.DataFrame(all_results).sort_values('F1_macro', ascending=False)

print(f"\n{'='*60}")
print(f"  Comparaison des modèles (27 classes)")
print(f"{'='*60}")
print(results_df.to_string(index=False))

best = results_df.iloc[0]
print(f"\n✅ Meilleur : {best['Model']} (F1-macro={best['F1_macro']:.4f})")

results_df.to_csv(MODELS_DIR / 'image_optim_comparison.csv', index=False)
print("✅ Résultats sauvegardés dans models/image_optim_comparison.csv")


  Comparaison des modèles (27 classes)
                         Model  F1_macro  F1_weighted
                      LightGBM  0.524211     0.576630
               Baseline (NB09)  0.517074     0.555750
                       XGBoost  0.515680     0.573161
Logistic Regression (weighted)  0.506625     0.550666
             Gradient Boosting  0.490881     0.557675
     Random Forest (optimized)  0.478902     0.546981
                      CatBoost  0.428086     0.481950

✅ Meilleur : LightGBM (F1-macro=0.5242)
✅ Résultats sauvegardés dans models/image_optim_comparison.csv
